<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">RAG PIPELINE: RETRIEVAL-AUGMENTED GENERATION</h2>

This guide demonstrates how to build a production-quality <strong>Retrieval-Augmented Generation (RAG)</strong> pipeline using <strong>Razer AIKit</strong>. RAG combines document retrieval with generative AI to answer questions based on a curated knowledge base, enabling local, privacy-preserving AI applications.

By following this guide, you will learn how to:

<ul>
  <li>Download and parse research papers from arXiv using <code>docling</code></li>
  <li>Chunk documents intelligently with <code>LangChain</code> text splitters</li>
  <li>Generate embeddings using <code>BAAI/bge-small-en-v1.5</code></li>
  <li>Store vectors persistently with <code>ChromaDB</code></li>
  <li>Perform semantic search and cross-encoder reranking</li>
  <li>Generate context-aware answers with <code>Qwen/Qwen3.5-4B</code></li>
  <li>Run all models simultaneously on local hardware</li>
</ul>

<strong>Architecture:</strong> Three models running simultaneously:
<ul>
  <li><strong>Port 8000:</strong> Embedding model (BAAI/bge-small-en-v1.5)</li>
  <li><strong>Port 8001:</strong> Chat model (Qwen/Qwen3.5-4B)</li>
  <li><strong>Port 8002:</strong> Reranker model (BAAI/bge-reranker-base)</li>
</ul>

<h3 style="color:#44D62C; text-align:left;">📥 1. Document Ingestion: Download & Parse arXiv Papers</h3>

First, we'll download 20 landmark LLM research papers from arXiv and parse them using <code>docling</code>, which preserves document structure (headings, sections, tables) better than basic PDF parsers.

In [ ]:
import requests
from pathlib import Path
import json
from tqdm import tqdm
import time

# Curated collection of 10 landmark LLM research papers
LANDMARK_PAPERS = {
    "1706.03762": {"title": "Attention Is All You Need", "year": 2017, "topic": "Transformer Architecture"},
    "2005.14165": {"title": "GPT-3", "year": 2020, "topic": "Few-Shot Learning"},
    "2203.02155": {"title": "InstructGPT", "year": 2022, "topic": "RLHF"},
    "2201.11903": {"title": "Chain-of-Thought Prompting", "year": 2022, "topic": "Reasoning"},
    "2302.13971": {"title": "LLaMA", "year": 2023, "topic": "Open Models"},
    "2104.09864": {"title": "LoRA", "year": 2021, "topic": "Efficient Fine-Tuning"},
    "2305.14314": {"title": "QLoRA", "year": 2023, "topic": "Quantization"},
    "2005.11401": {"title": "RAG Paper", "year": 2020, "topic": "RAG"},
    "2212.10496": {"title": "E5 Embeddings", "year": 2022, "topic": "Embeddings"},
    "2303.08774": {"title": "GPT-4 Technical Report", "year": 2023, "topic": "Multimodal"},
}

def download_arxiv_papers(papers_dict, output_dir="/var/aikit/rag-pipeline/documents"):
    """Download curated collection of landmark LLM papers from arXiv."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    metadata_list = []
    successful = 0
    failed = []
    
    print(f"📥 Downloading {len(papers_dict)} landmark LLM papers from arXiv...")
    
    for arxiv_id, info in tqdm(papers_dict.items(), desc="Downloading papers"):
        try:
            url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            
            filename = f"{arxiv_id}_{info['title'][:30].replace(':', '').replace('/', '_')}.pdf"
            filepath = output_path / filename
            filepath.write_bytes(response.content)
            
            metadata_list.append({
                "arxiv_id": arxiv_id,
                "filename": filename,
                "title": info["title"],
                "year": info["year"],
                "topic": info["topic"],
                "file_size_mb": len(response.content) / (1024 * 1024)
            })
            
            successful += 1
            time.sleep(1)  # Be nice to arXiv servers
            
        except Exception as e:
            failed.append({"arxiv_id": arxiv_id, "error": str(e)})
            print(f"\n❌ Failed to download {arxiv_id}: {e}")
    
    # Save metadata
    metadata_path = output_path / "metadata.json"
    with open(metadata_path, "w") as f:
        json.dump({
            "papers": metadata_list,
            "failed": failed,
            "total_downloaded": successful,
            "total_size_mb": sum(m["file_size_mb"] for m in metadata_list)
        }, f, indent=2)
    
    print(f"\n✅ Successfully downloaded {successful}/{len(papers_dict)} papers")
    print(f"📊 Total size: {sum(m['file_size_mb'] for m in metadata_list):.1f} MB")
    print(f"📄 Metadata saved to: {metadata_path}")
    
    return metadata_list

# Download papers
metadata = download_arxiv_papers(LANDMARK_PAPERS)

Now parse the downloaded PDFs using <strong>docling</strong>, which extracts text while preserving document structure.

In [ ]:
from docling.document_converter import DocumentConverter
from pathlib import Path
import json
from tqdm import tqdm

# Initialize docling converter
converter = DocumentConverter()

documents_dir = Path("/var/aikit/rag-pipeline/documents")
processed_dir = Path("/var/aikit/rag-pipeline/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

parsed_documents = []

print("📄 Parsing PDFs with docling...")
pdf_files = sorted(documents_dir.glob("*.pdf"))

for pdf_file in tqdm(pdf_files, desc="Parsing papers"):
    try:
        result = converter.convert(str(pdf_file))
        markdown_text = result.document.export_to_markdown()
        
        # Extract arxiv_id from filename
        arxiv_id = pdf_file.stem.split('_')[0]
        
        parsed_documents.append({
            "arxiv_id": arxiv_id,
            "filename": pdf_file.name,
            "text": markdown_text,
            "char_count": len(markdown_text)
        })
        
    except Exception as e:
        print(f"\n⚠️  Error parsing {pdf_file.name}: {e}")

print(f"\n✅ Parsed {len(parsed_documents)} papers")
print(f"📊 Total characters: {sum(d['char_count'] for d in parsed_documents):,}")

<h3 style="color:#44D62C; text-align:left;">✂️ 2. Text Chunking with LangChain</h3>

Split documents into chunks for optimal retrieval. We use <code>RecursiveCharacterTextSplitter</code> with:
<ul>
  <li><strong>chunk_size=450</strong>: ~2-3 paragraphs per chunk</li>
  <li><strong>chunk_overlap=50</strong>: 20% overlap prevents context loss at boundaries</li>
  <li><strong>Hierarchical separators</strong>: Prioritizes natural boundaries (paragraphs > sentences > words)</li>
</ul>

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
    is_separator_regex=False
)

all_chunks = []

print("✂️  Chunking documents...")

for doc in tqdm(parsed_documents, desc="Chunking"):
    # Find metadata for this paper
    paper_meta = next((m for m in metadata if m["arxiv_id"] == doc["arxiv_id"]), {})
    
    # Create LangChain documents
    chunks = text_splitter.create_documents(
        texts=[doc["text"]],
        metadatas=[{
            "source": doc["arxiv_id"],
            "title": paper_meta.get("title", "Unknown"),
            "year": paper_meta.get("year", 0),
            "topic": paper_meta.get("topic", "Unknown")
        }]
    )
    
    # Add chunk index to metadata
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_index"] = i
    
    all_chunks.extend(chunks)

print(f"\n✅ Created {len(all_chunks)} chunks")
print(f"📊 Average chunk size: {sum(len(c.page_content) for c in all_chunks) / len(all_chunks):.0f} characters")

# Save chunks for reference
chunks_data = [{
    "text": chunk.page_content,
    "metadata": chunk.metadata
} for chunk in all_chunks]

with open(processed_dir / "chunks.json", "w") as f:
    json.dump(chunks_data, f, indent=2)

print(f"💾 Chunks saved to {processed_dir / 'chunks.json'}")

<h3 style="color:#44D62C; text-align:left;">🧠 3. Generate Embeddings with BAAI/bge-small-en-v1.5</h3>

First, start the embedding model on <strong>port 8000</strong>.

In [ ]:
# Download and start embedding model
!rzr-aikit model download BAAI/bge-small-en-v1.5
!rzr-aikit model run BAAI/bge-small-en-v1.5 --port 8000 --gpu-memory-utilization 0.1

Now generate embeddings for all chunks using the OpenAI-compatible API.

In [ ]:
from openai import OpenAI
from tqdm import tqdm

# Connect to embedding model
embedding_client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="rzr-aikit"
)

def embed_batch(texts, batch_size=32):
    """Batch embed texts for efficiency."""
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i+batch_size]
        response = embedding_client.embeddings.create(
            model="BAAI/bge-small-en-v1.5",
            input=batch
        )
        embeddings.extend([item.embedding for item in response.data])
    return embeddings

# Embed all chunks
chunk_texts = [chunk.page_content for chunk in all_chunks]
print(f"\n🔢 Embedding {len(chunk_texts)} chunks...")
embeddings = embed_batch(chunk_texts)

print(f"\n✅ Generated {len(embeddings)} embeddings")
print(f"📊 Embedding dimension: {len(embeddings[0])}")

<h3 style="color:#44D62C; text-align:left;">💾 4. Store Vectors in ChromaDB</h3>

Create a persistent ChromaDB collection to store embeddings with metadata for efficient semantic search.

In [ ]:
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB with persistent storage
client = chromadb.PersistentClient(
    path="/var/aikit/rag-pipeline/vector_db",
    settings=Settings(anonymized_telemetry=False, allow_reset=True)
)

# Create or get collection
collection = client.get_or_create_collection(
    name="arxiv_papers",
    metadata={
        "description": "Landmark LLM research papers",
        "embedding_model": "BAAI/bge-small-en-v1.5",
        "embedding_dim": 384,
        "chunk_size": 450,
        "chunk_overlap": 50
    }
)

print(f"📦 ChromaDB collection: {collection.name}")
print(f"📊 Current count: {collection.count()} documents")

In [ ]:
# Prepare data for ChromaDB
chunk_metadatas = [chunk.metadata for chunk in all_chunks]
chunk_ids = [f"chunk_{i}" for i in range(len(all_chunks))]

# Add to ChromaDB (only if collection is empty)
if collection.count() == 0:
    print("💾 Adding documents to ChromaDB...")
    collection.add(
        documents=chunk_texts,
        embeddings=embeddings,
        metadatas=chunk_metadatas,
        ids=chunk_ids
    )
    print(f"\n✅ Added {collection.count()} chunks to vector database")
else:
    print(f"ℹ️  Collection already contains {collection.count()} documents (skipping insert)")

<h3 style="color:#44D62C; text-align:left;">🔍 5. Retrieval Pipeline: Semantic Search</h3>

Implement semantic search to find relevant chunks for a query.

In [ ]:
def retrieve_chunks(query, top_k=10, metadata_filter=None):
    """
    Retrieve most relevant chunks for a query.
    
    Args:
        query: User query string
        top_k: Number of results to return
        metadata_filter: Optional ChromaDB where filter
    
    Returns:
        List of dicts with 'text', 'metadata', 'distance'
    """
    # Embed query
    query_embedding = embedding_client.embeddings.create(
        model="BAAI/bge-small-en-v1.5",
        input=query
    ).data[0].embedding
    
    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=metadata_filter,
        include=["documents", "metadatas", "distances"]
    )
    
    # Format results
    formatted_results = []
    for i in range(len(results['documents'][0])):
        formatted_results.append({
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'distance': results['distances'][0][i]
        })
    
    return formatted_results

# Test retrieval
test_query = "What is the attention mechanism in transformers?"
print(f"🔍 Test query: {test_query}\n")

results = retrieve_chunks(test_query, top_k=5)

print("📚 Top 5 results:")
for i, result in enumerate(results):
    print(f"\n{i+1}. {result['metadata']['title']} (distance: {result['distance']:.3f})")
    print(f"   {result['text'][:150]}...")

<h3 style="color:#44D62C; text-align:left;">🎯 6. Cross-Encoder Reranking</h3>

Use a cross-encoder model to rerank retrieved chunks for better relevance. Cross-encoders see the query and document together, enabling more accurate scoring than bi-encoder embeddings.

In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np

# Load cross-encoder reranker
print("📥 Loading cross-encoder reranker...")
reranker = CrossEncoder('BAAI/bge-reranker-base')
print("✅ Reranker loaded")

def rerank_results(query, chunks, top_k=5):
    """
    Rerank retrieved chunks using cross-encoder.
    
    Cross-encoder sees query + document together for better relevance scoring.
    """
    # Prepare query-document pairs
    pairs = [[query, chunk['text']] for chunk in chunks]
    
    # Score pairs
    scores = reranker.predict(pairs)
    
    # Rerank by score (descending)
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    
    reranked = [chunks[i] for i in ranked_indices]
    reranked_scores = scores[ranked_indices]
    
    return reranked, reranked_scores

# Test reranking
print(f"\n🎯 Reranking top 10 results to top 5...\n")
candidates = retrieve_chunks(test_query, top_k=10)
reranked_chunks, reranked_scores = rerank_results(test_query, candidates, top_k=5)

print("📚 Top 5 after reranking:")
for i, (chunk, score) in enumerate(zip(reranked_chunks, reranked_scores)):
    print(f"\n{i+1}. {chunk['metadata']['title']} (score: {score:.3f})")
    print(f"   {chunk['text'][:150]}...")

<h3 style="color:#44D62C; text-align:left;">💬 7. Answer Generation with RAG</h3>

Start the chat model and implement the full RAG pipeline: retrieve → rerank → generate.

In [ ]:
# Download and start chat model on port 8001
!rzr-aikit model download Qwen/Qwen3.5-4B
!rzr-aikit model run Qwen/Qwen3.5-4B --port 8001 --gpu-memory-utilization 0.7 --max-model-len 180000

In [ ]:
# Connect to chat model
chat_client = OpenAI(
    base_url="http://localhost:8001/v1",
    api_key="rzr-aikit"
)

RAG_SYSTEM_PROMPT = """You are a helpful AI assistant that answers questions based on provided research paper excerpts.

Instructions:
- Use ONLY the information from the provided context
- Cite specific papers when making claims (e.g., "According to the Attention paper...")
- If the context lacks information, say "I don't have enough information in the provided papers"
- Be concise and accurate
- Acknowledge uncertainty when appropriate"""

def rag_query(query, top_k=5, use_reranking=True):
    """
    Full RAG pipeline: retrieve → rerank → generate.
    
    Args:
        query: User question
        top_k: Number of chunks to use as context
        use_reranking: Whether to use cross-encoder reranking
    
    Returns:
        Dict with answer, sources, and relevance scores
    """
    # 1. Retrieve candidates
    candidates = retrieve_chunks(query, top_k=10)
    
    # 2. Rerank (optional)
    if use_reranking:
        final_chunks, scores = rerank_results(query, candidates, top_k=top_k)
    else:
        final_chunks = candidates[:top_k]
        scores = np.array([1 - c['distance'] for c in final_chunks])
    
    # 3. Format context with source attribution
    context_parts = []
    for i, chunk in enumerate(final_chunks):
        source = f"[Paper {i+1}: {chunk['metadata']['title']} ({chunk['metadata']['year']})]" 
        context_parts.append(f"{source}\n{chunk['text']}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # 4. Generate answer
    response = chat_client.chat.completions.create(
        model="Qwen/Qwen3.5-4B",
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ],
        temperature=0.1,
        max_tokens=512,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )
    
    return {
        "answer": response.choices[0].message.content,
        "sources": final_chunks,
        "relevance_scores": scores.tolist() if isinstance(scores, np.ndarray) else scores
    }

# Test RAG pipeline
test_question = "What are scaling laws for language models?"
print(f"❓ Question: {test_question}\n")

result = rag_query(test_question)

print(f"🤖 Answer:\n{result['answer']}\n")
print(f"📚 Sources Used:")
for i, source in enumerate(result['sources']):
    print(f"  {i+1}. {source['metadata']['title']} (relevance: {result['relevance_scores'][i]:.3f})")

<h3 style="color:#44D62C; text-align:left;">🎮 8. Interactive RAG Query Interface</h3>

Try your own questions! The pipeline will retrieve relevant papers, rerank them, and generate an answer.

In [ ]:
# Interactive RAG query
print("💡 Example questions you can ask:")
print("  - What is the attention mechanism in transformers?")
print("  - What is retrieval-augmented generation?")
print("  - Compare GPT-3 and InstructGPT training methods")
print("  - What are the benefits of LoRA for fine-tuning?")
print("\n" + "="*70 + "\n")

user_query = input("Enter your question: ")

if user_query:
    print(f"\n🔍 Processing query: {user_query}\n")
    result = rag_query(user_query)
    
    print(f"🤖 Answer:\n{result['answer']}\n")
    print(f"📚 Sources:")
    for i, source in enumerate(result['sources']):
        score = result['relevance_scores'][i]
        print(f"  {i+1}. {source['metadata']['title']} - {source['metadata']['topic']} ({source['metadata']['year']}) [relevance: {score:.3f}]")

<h3 style="color:#44D62C; text-align:left;">⚖️ With vs. Without Reranking</h3>

Compare answer quality with and without cross-encoder reranking.

In [ ]:
comparison_query = "How does LoRA work for fine-tuning?"

print(f"🔬 Comparing with/without reranking for: {comparison_query}\n")
print("="*70)

# Without reranking
print("\n🔵 WITHOUT Reranking (embedding similarity only):\n")
result_no_rerank = rag_query(comparison_query, use_reranking=False)
print(f"Answer: {result_no_rerank['answer'][:300]}...\n")
print("Sources:")
for i, src in enumerate(result_no_rerank['sources'][:3]):
    print(f"  {i+1}. {src['metadata']['title']}")

# With reranking
print("\n" + "="*70)
print("\n🟢 WITH Reranking (cross-encoder):\n")
result_with_rerank = rag_query(comparison_query, use_reranking=True)
print(f"Answer: {result_with_rerank['answer'][:300]}...\n")
print("Sources:")
for i, src in enumerate(result_with_rerank['sources'][:3]):
    score = result_with_rerank['relevance_scores'][i]
    print(f"  {i+1}. {src['metadata']['title']} (score: {score:.3f})")

<h3 style="color:#44D62C; text-align:left;">✅ Summary</h3>

You've successfully built a production-quality RAG pipeline using Razer AIKit!

##### Key accomplishments:

- ✅ Downloaded and parsed 10 landmark LLM research papers from arXiv
- ✅ Chunked documents intelligently with LangChain (450 chars, 50 overlap)
- ✅ Generated 384-dim embeddings with BAAI/bge-small-en-v1.5
- ✅ Stored vectors persistently in ChromaDB (~600-1000 chunks)
- ✅ Implemented semantic search with metadata filtering
- ✅ Added cross-encoder reranking for higher quality
- ✅ Generated context-aware answers with Qwen 3.5 4B
- ✅ Ran all three models simultaneously on local hardware

##### Architecture recap:

- <strong>Port 8000:</strong> BAAI/bge-small-en-v1.5 (embedding model)
- <strong>Port 8001:</strong> Qwen/Qwen3.5-4B (chat model)
- <strong>Port 8002:</strong> BAAI/bge-reranker-base (reranking model)

##### Next steps:

- <strong>No-code alternative:</strong> Run this same pipeline through Open WebUI — see [docs/rag.md](../docs/rag.md)
- <strong>Add more documents:</strong> Expand your knowledge base
- <strong>Hybrid search:</strong> Combine semantic + BM25 keyword search
- <strong>Multi-turn chat:</strong> Add conversation history
- <strong>Web UI:</strong> Build a Gradio or Streamlit interface
- <strong>Evaluation:</strong> Create test sets to measure answer quality

All models run locally — <strong>fast, private, and cloud-free</strong>. You're now ready to build advanced RAG applications with Razer AIKit!

<h3 style="color:#44D62C; text-align:left;">🛑 Cleanup: Stop Models</h3>

In [ ]:
# Stop all running models
!rzr-aikit model stop